# AI403 Deep Learning Final Project – run in Google Colab
**Noise-robust environmental sound classification (ESC-50)**

**Before you start:** `Runtime → Change runtime type → T4 GPU` (the code uses the GPU automatically; it also runs on CPU, just slower).

Two ways to use this notebook:
* **Mode A – quick (no training, ~5 min):** reuse the results shipped in the repo → rebuild tables/figures → launch the live demo. Run cells 1–3, then **A1–A2**.
* **Mode B – full reproduction:** download ESC-50, retrain everything (resumable). Run cells 1–3, then **B1–B4**.

⚠️ Re-training on a different machine gives *slightly different* numbers than the ones in the submitted report (different hardware/RNG). If you retrain, re-check every number in your report/slides before submitting.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU - running on CPU'
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Get the project
Upload `project_repo.zip` when prompted (or put it in `/content` first). If you pushed to GitHub, you can use `!git clone <your-repo-url> /content/project` instead and skip the unzip.

In [ ]:
import os, zipfile
if not os.path.exists('/content/project'):
    if not os.path.exists('/content/project_repo.zip'):
        from google.colab import files; files.upload()          # choose project_repo.zip
    zipfile.ZipFile('/content/project_repo.zip').extractall('/content/project')
%cd /content/project
!ls

## 3. Install the few extra packages (torch, numpy, librosa are already on Colab)

In [ ]:
!pip -q install gradio pyyaml tabulate reportlab pypdf pillow

---
# Mode A – quick: reuse shipped results + demo

**A1. Rebuild all tables & figures from the saved results** (`experiments/results/`)

In [ ]:
!python scripts/aggregate.py > /dev/null && echo 'tables/figures rebuilt' && cat experiments/tables/table4_main_results.md

In [ ]:
from IPython.display import Image, display
for f in ['robustness.png','val_curves.png','hparam_lr.png','per_class_f1.png']:
    display(Image('experiments/figures/'+f, width=560))

**A2. Live demo** – prints a public `gradio.live` link; open it, upload/record a sound, move the noise slider. Stop the cell to end the demo.

In [ ]:
!python demo/app.py

---
# Mode B – full reproduction (retrain everything)

**B1. Download + preprocess ESC-50** (~600 MB download, a few minutes)

In [ ]:
!python data/prepare_esc50.py

**B2. (Optional but recommended) keep results safe on Google Drive** so a disconnect does not lose them. The runner is resumable: re-run B3 and finished runs are skipped.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
SAVE='/content/drive/MyDrive/dl_project_results'; os.makedirs(SAVE, exist_ok=True)
!rm -rf experiments/results && ln -s {SAVE} experiments/results
!ls experiments/results | head

**B3. Run every experiment** (tuning grid, 3-seed finals, data-strategy, SSL pre-training, ablations). Takes a while; resumable.

In [ ]:
!python scripts/run_all.py all

**B4. Baseline, tables/figures, demo checkpoint**

In [ ]:
!python -m src.models.baseline_ml
!python scripts/aggregate.py > /dev/null
!python scripts/export_demo_checkpoint.py
!cat experiments/tables/table4_main_results.md

---
## Download your results

In [ ]:
!zip -qr /content/results_and_figures.zip experiments checkpoints -x '*_logits.npy'
from google.colab import files; files.download('/content/results_and_figures.zip')